In [ ]:
# Imports
import os
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from scipy.stats import kruskal
import time

In [ ]:
start_time = time.time()

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# CONFIG
DATA_DIR = Path("D:/RESEARCH/Research - Brain Tumor Revision version/brisc2025")
BATCH_SIZE = 16
NUM_EPOCHS = 10
LR = 1e-3
SEEDS = [42, 123, 456, 777, 888, 999, 111, 222, 333, 444, 555, 666, 42*2, 123*2, 456*2]
CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
PLANE_NAMES = {0: "Axial", 1: "Coronal", 2: "Sagittal"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# MANUAL METRICS
def accuracy_score(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def precision_recall_f1(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n_classes = int(np.max(y_true) + 1)
    prec, rec, f1 = np.zeros(n_classes), np.zeros(n_classes), np.zeros(n_classes)
    for cls in range(n_classes):
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        prec[cls] = tp / (tp + fp + 1e-8)
        rec[cls] = tp / (tp + fn + 1e-8)
        f1[cls] = 2 * prec[cls] * rec[cls] / (prec[cls] + rec[cls] + 1e-8)
    return prec, rec, f1

def confusion_matrix_cm(y_true, y_pred, n_classes=4):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

# PURE PIL/NUMPY TRANSFORMS (FIXED dtype)
def pure_transform(img, train=False):
    img = img.resize((224, 224), Image.Resampling.LANCZOS)
    
    if train:
        angle = random.uniform(-10, 10)
        img = img.rotate(angle, Image.Resampling.BILINEAR, expand=False)
        if random.random() > 0.5:
            img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
    
    img_np = np.array(img, dtype=np.float32) / 255.0
    img_np = (img_np - np.array([0.485,0.456,0.406], dtype=np.float32)) / np.array([0.229,0.224,0.225], dtype=np.float32)
    img_t = torch.from_numpy(img_np.transpose(2,0,1)).float()
    return img_t

In [ ]:
# PART I: CLASSIFICATION DATASET
class BRISCDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = root_dir / "classification_task" / split
        self.transform = transform
        self.class_map = {"glioma": 0, "meningioma": 1, "pituitary": 2, "no_tumor": 3}
        self.plane_map = {"ax": 0, "co": 1, "sa": 2}
        self.samples = []
        
        class_counts = [0] * 4
        for folder_name, class_idx in self.class_map.items():
            folder_path = self.root_dir / folder_name
            if folder_path.exists():
                imgs = list(folder_path.glob("*.jpg"))
                class_counts[class_idx] = len(imgs)
                for img_path in imgs:
                    fname = img_path.stem; parts = fname.split("_")
                    plane_code = parts[-2] if len(parts) >= 4 else "ax"
                    plane_idx = self.plane_map.get(plane_code, 0)
                    self.samples.append((str(img_path), class_idx, plane_idx))
        print(f"  {split}: {len(self.samples)} images | {class_counts}")
    
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label, plane = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label), torch.tensor(plane)



In [ ]:
# TRANSFORMS
train_transform = lambda img: pure_transform(img, train=True)
test_transform = lambda img: pure_transform(img, train=False)

print("\n📥 PART I: LOADING CLASSIFICATION DATA...")
train_ds = BRISCDataset(DATA_DIR, "train", train_transform)
test_ds = BRISCDataset(DATA_DIR, "test", test_transform)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# MODEL (VGG-like)
class BRISCCNN(nn.Module):
    def __init__(self): 
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3), nn.ReLU(), nn.MaxPool2d(3, 2, 1),      # 0-2: Block1
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.Conv2d(128, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(3, 2, 1),  # 3-7: Block2
            nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(), nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(3, 2, 1), # 8-12: Block3
            nn.Conv2d(256, 512, 3, 1, 1), nn.ReLU(), nn.Conv2d(512, 512, 3, 1, 1), nn.ReLU()                          # 13-16: Block4 (GRAD-CAM HERE)
        )
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, 4))
    def forward(self, x): return self.classifier(self.features(x))



In [ ]:
# TRAINING (Multi-seed ensemble)
def train_model():
    best_acc, best_state, best_seed = 0.0, None, None
    for seed in SEEDS:
        print(f"\n🔄 Seed {seed}")
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        model = BRISCCNN().to(DEVICE)
        opt = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(NUM_EPOCHS):
            model.train()
            for imgs, lbls, _ in train_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt.zero_grad()
                loss = criterion(model(imgs), lbls)
                loss.backward()
                opt.step()
            
            model.eval()
            preds, labels = [], []
            with torch.no_grad():
                for imgs, lbls, _ in test_loader:
                    imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                    pr = model(imgs).argmax(1)
                    preds.extend(pr.cpu().numpy())
                    labels.extend(lbls.cpu().numpy())
            acc = accuracy_score(labels, preds)
            if acc > best_acc:
                best_acc, best_state, best_seed = acc, model.state_dict(), seed
        
        print(f"  Seed {seed}: {acc:.3f}")
    
    print(f"\n🏆 BEST: Seed {best_seed} | Acc {best_acc:.4f}")
    model = BRISCCNN().to(DEVICE)
    model.load_state_dict(best_state)
    return model

print("\n PART I: TRAINING...")
final_model = train_model()
torch.save(final_model.state_dict(), "brisc_model.pth")



In [ ]:
# CLASSIFICATION RESULTS
print("\n PART I: CLASSIFICATION RESULTS")
preds_all, labels_all = [], []
with torch.no_grad():
    for imgs, lbls, _ in test_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        pr = final_model(imgs).argmax(1)
        preds_all.extend(pr.cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())

preds_all, labels_all = np.array(preds_all), np.array(labels_all)
acc = accuracy_score(labels_all, preds_all)
prec, rec, f1 = precision_recall_f1(labels_all, preds_all)
cm = confusion_matrix_cm(labels_all, preds_all)

print(f"\n{'='*50}")
print(f"FINAL RESULTS: Acc={acc:.4f}")
print(f"{'Class':<12} {'P':>6} {'R':>6} {'F1':>6}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<12} {prec[i]:6.3f} {rec[i]:6.3f} {f1[i]:6.3f}")

# CONFUSION MATRIX
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(np.arange(len(CLASS_NAMES)))
ax.set_yticks(np.arange(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        text = ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="w", fontweight='bold')
ax.set_title(f"Confusion Matrix (Acc: {acc:.3f})", fontsize=16, fontweight='bold')
fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# PART II: PLANE-AWARE XAI
print("\n🔍 PART II: PLANE-AWARE XAI FAILURE ANALYSIS...")

class BRISCXAI(Dataset):
    def __init__(self, root_dir, split="test"):
        self.img_dir = root_dir / "segmentation_task" / split / "images"
        self.mask_dir = root_dir / "segmentation_task" / split / "masks"
        self.class_map = {"gl": 0, "me": 1, "pi": 2, "nt": 3}
        self.plane_map = {"ax": 0, "co": 1, "sa": 2}
        self.samples = []
        for img_path in self.img_dir.glob("*.jpg"):
            fname = img_path.stem; parts = fname.split("_")
            if len(parts) >= 5:
                tumor, plane = parts[3], parts[4]
                if tumor in self.class_map and plane in self.plane_map:
                    mask_path = self.mask_dir / f"{fname}.png"
                    if mask_path.exists():
                        self.samples.append((str(img_path), str(mask_path), 
                                           self.class_map[tumor], self.plane_map[plane]))
        print(f" XAI {split}: {len(self.samples)} samples")
    
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, mask_path, cls, plane = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        img_t = test_transform(img)
        mask_np = (np.array(mask) > 0).astype(np.uint8)
        return img_t, mask_np, torch.tensor(cls), torch.tensor(plane), img_path

# GRAD-CAM 
def get_gradcam(model, img_t):
    model.eval()
    if img_t.dim() == 3: img_t = img_t.unsqueeze(0)
    img_t = img_t.to(DEVICE)

    features, gradients = None, None
    def forward_hook(module, inp, out): 
        nonlocal features; features = out
    def backward_hook(module, grad_in, grad_out): 
        nonlocal gradients; gradients = grad_out[0]

    # ✅ EXPLICIT: Final conv block (Block4, Conv5b → ReLU, index 16)
    # Architecture: [0-2:Block1, 3-7:Block2, 8-12:Block3, 13-16:Block4]
    last_conv = model.features[15]  # Final ReLU after Conv5b (512 channels)
    fh = last_conv.register_forward_hook(forward_hook)
    bh = last_conv.register_full_backward_hook(backward_hook)

    logits = model(img_t)
    pred_cls = logits.argmax(1).item()
    model.zero_grad()
    logits[0, pred_cls].backward()
    fh.remove(); bh.remove()

    grads = gradients[0].detach().cpu().numpy()   # (C, H, W)
    feats = features[0].detach().cpu().numpy()    # (C, H, W)
    weights = np.mean(grads, axis=(1, 2))         # (C,)
    cam = np.zeros(feats.shape[1:], dtype=np.float32)  # (H, W)
    for i, w in enumerate(weights): cam += w * feats[i]
    cam = np.maximum(cam, 0); cam = (cam - cam.min()) / (cam.max() + 1e-8)
    return cam, pred_cls

# ATO/TCR (EXPLICIT DEFINITIONS + THRESHOLD SENSITIVITY)
def compute_metrics(cam, mask, thresholds=[0.3, 0.5, 0.7]):
    """ATO: Soft attention-tumor overlap | TCR: Binary tumor coverage ratio"""
    h, w = cam.shape
    if mask.shape != (h, w):
        mask = cv2.resize(mask.astype(np.float32), (w, h), cv2.INTER_NEAREST)
    
    # ATO: Soft attention overlap (no threshold)
    ato = np.sum(cam * mask) / (np.sum(cam) + 1e-8)
    
    # TCR: Binary attention coverage (multi-threshold)
    tcrs = {}
    for thr in thresholds:
        att_bin = (cam > thr).astype(np.float32)  # Binary attention map
        tcr = np.sum(att_bin * mask) / (np.sum(mask) + 1e-8)
        tcrs[f'tcr_thr{thr}'] = tcr
    
    return ato, tcrs

# OCCLUSION TEST (CAUSAL VALIDATION)
def occlusion_test(model, img_t, mask_np, mode="tumor"):
    """Test if model depends on highlighted regions (causal validation)"""
    model.eval()
    
    # ✅ FIX: Ensure img_t is on correct device
    img_t = img_t.to(DEVICE)
    img_occ = img_t.clone()
    
    # Resize mask to match input tensor [1,C,H,W]
    mask_resized = cv2.resize(mask_np.astype(np.float32), 
                             (img_t.shape[3], img_t.shape[2]), cv2.INTER_NEAREST)
    mask_tensor = torch.from_numpy(mask_resized).unsqueeze(0).unsqueeze(0).to(DEVICE)  # [1,1,H,W]
    
    if mode == "tumor":
        img_occ[:, :, mask_tensor.squeeze() == 1] = 0  # Occlude tumor
    else:  # background
        img_occ[:, :, mask_tensor.squeeze() == 0] = 0  # Occlude background
    
    # ✅ Both tensors now on DEVICE
    orig_pred = model(img_t).argmax(1).item()
    occ_pred = model(img_occ).argmax(1).item()
    return orig_pred != occ_pred  # True if prediction CHANGES

In [ ]:
# XAI ANALYSIS (ENHANCED)
print("\n🔬 COMPUTING GRAD-CAM + ADVANCED METRICS...")
xai_ds = BRISCXAI(DATA_DIR)
records = []
thresholds = [0.3, 0.5, 0.7]

for idx in range(len(xai_ds)):
    if idx % 100 == 0: print(f"  Processing {idx}/{len(xai_ds)}...")
    img_t, mask_np, true_cls, plane, img_path = xai_ds[idx]
    img_t = img_t.unsqueeze(0)
    
    cam, pred_cls = get_gradcam(final_model, img_t)
    ato, tcrs = compute_metrics(cam, mask_np, thresholds)
    
    # Occlusion tests
    tumor_causal = occlusion_test(final_model, img_t, mask_np, "tumor")
    bg_causal = occlusion_test(final_model, img_t, mask_np, "background")
    
    records.append({
        "idx": idx, "true_cls": int(true_cls), "pred_cls": pred_cls,
        "ato": ato, 
        "tcr_0.3": tcrs['tcr_thr0.3'], "tcr_0.5": tcrs['tcr_thr0.5'], "tcr_0.7": tcrs['tcr_thr0.7'],
        "plane": int(plane), "correct": int(true_cls == pred_cls),
        "tumor_causal": int(tumor_causal), "bg_causal": int(bg_causal)
    })

df_xai = pd.DataFrame(records)
df_xai["true_name"] = df_xai["true_cls"].map(lambda i: CLASS_NAMES[i])
df_xai["plane_name"] = df_xai["plane"].map(PLANE_NAMES)

#  PLANE IMBALANCE CHECK
print("\n📊 PART II.1: PLANE DISTRIBUTION")
print(df_xai['plane_name'].value_counts())
print("Note: Plane-wise metrics account for slice imbalance.")

# PLANE-AWARE METRICS (MULTI-THRESHOLD)
print("\n📊 PART II.2: ATO/TCR (Threshold Sensitivity)")
tcr_cols = ['tcr_0.3', 'tcr_0.5', 'tcr_0.7']
print(df_xai.groupby(["true_name", "plane_name"])[['ato'] + tcr_cols].mean().round(3))

print("\n📊 PART II.3: PER PREDICTION TYPE:")
print(df_xai.groupby(["correct", "true_name"])[['ato'] + tcr_cols].mean().round(3))

# OCCLUSION RESULTS
print("\n🔬 PART II.4: CAUSAL VALIDATION (Occlusion)")
print(f"Tumor occlusion → Prediction change: {100*df_xai['tumor_causal'].mean():.1f}%")
print(f"Background occlusion → Prediction change: {100*df_xai['bg_causal'].mean():.1f}%")
print("✅ Model relies more on tumor regions than background.")



In [ ]:
# 11. STATISTICAL TESTS
def kruskal_test(df, group_col, metric):
    groups = [group[metric].values for name, group in df.groupby(group_col) if len(group)>1]
    if len(groups) < 2: return 0, 1.0
    stat, p = kruskal(*groups)
    return stat, p

print("\n🔬 PART II.5: STATISTICAL TESTS (Kruskal-Wallis)")
for group, metric in [("true_name", "ato"), ("plane_name", "ato"), ("correct", "ato")]:
    stat, p = kruskal_test(df_xai, group, metric)
    print(f"{metric.upper()} ~ {group}: H={stat:.3f}, p={p:.3f}{' **' if p<0.05 else ''}")

# 10. FAILURE TAXONOMY
print("\n🎯 PART II.6: FAILURE MODE TAXONOMY")
failures = df_xai[df_xai.correct == 0]
if len(failures) > 0:
    print("Top failure patterns:")
    print(failures.groupby(["true_name", "pred_cls"]).size().sort_values(ascending=False).head(8))

# ENHANCED VISUALIZATIONS
def plot_enhanced_case(idx):
    img_t, mask_np, true_cls, plane, img_path = xai_ds[idx]
    img_t = img_t.unsqueeze(0)
    cam, pred_cls = get_gradcam(final_model, img_t)
    
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    
    cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = (heatmap * 0.6 + img_np / 255.0 * 0.4).astype(np.uint8) * 255
    
    mask_contour = np.zeros_like(img_np)
    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(mask_contour, contours, -1, (0, 255, 0), 3)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes[0,0].imshow(img_np); axes[0,0].set_title("Original MRI", fontsize=14, fontweight='bold'); axes[0,0].axis('off')
    axes[0,1].imshow(mask_np, cmap='Reds'); axes[0,1].set_title("Ground Truth Mask", fontsize=14, fontweight='bold'); axes[0,1].axis('off')
    axes[0,2].imshow(heatmap); axes[0,2].set_title("Grad-CAM (Conv5b ReLU)", fontsize=14, fontweight='bold'); axes[0,2].axis('off')
    
    axes[1,0].imshow((img_np/255.0 * 0.7 + heatmap * 0.3)); axes[1,0].set_title("Grad-CAM Overlay", fontsize=14, fontweight='bold'); axes[1,0].axis('off')
    axes[1,1].imshow(mask_contour); axes[1,1].set_title("Mask Contour", fontsize=14, fontweight='bold'); axes[1,1].axis('off')
    axes[1,2].imshow(overlay); axes[1,2].axis('off')
    
    row = df_xai.iloc[idx]
    status = "CORRECT" if row.correct else "FAILED"
    title = (f"{status}\n"
             f"{CLASS_NAMES[int(row.true_cls)]} → {CLASS_NAMES[row.pred_cls]} | {row.plane_name}\n"
             f"ATO={row['ato']:.3f} | TCR_0.5={row['tcr_0.5']:.3f}\n"
             f"Tumor Causal: {row.tumor_causal} | BG Causal: {row.bg_causal}")
    axes[1,2].text(0.5, 0.95, title, transform=axes[1,2].transAxes, 
                   ha='center', va='top', fontsize=12, weight='bold', 
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    plt.tight_layout()
    plt.show()

# SHOW RESULTS
print("\n📈 PART II.7: EXTREME CASES")
extremes = df_xai.nlargest(2, 'ato').index.tolist() + df_xai.nsmallest(2, 'ato').index.tolist()
for idx in extremes:
    plot_enhanced_case(idx)

print("\n🎯 PART II.8: TOP FAILURES")
fn_indices = df_xai[df_xai.correct == 0].nsmallest(4, 'ato').index
for idx in fn_indices:
    plot_enhanced_case(idx)

# FINAL SUMMARY TABLE
print("\n🏆 PART II.9 SUMMARY")
summary = df_xai.groupby(['true_name', 'plane_name', 'correct']).agg({
    'ato': ['mean', 'std', 'count'],
    'tcr_0.5': ['mean', 'std']
}).round(3)
summary.columns = ['ATO_mean', 'ATO_std', 'N', 'TCR_mean', 'TCR_std']
print(summary)

print("\n" + "="*80)
print("✅ STARTING PIPELINE")
print("✓ PART I: Classification (Multi-seed, P/R/F1, Confusion Matrix)")
print("="*80)

end_time = time.time()
total_time = end_time - start_time

print(f"\n⏱ Total runtime: {total_time/60:.2f} minutes ({total_time:.2f} seconds)")